<a href="https://colab.research.google.com/github/tasamu61/MyAiStudyPre/blob/main/9.NeuralNetwork/09-1TwoLayerNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2層ニューラルネットを作ってみる

5章（`05-3numpy3.ipynb`）で作った「|」「-」「/」の3クラス判定は、1層（入力→出力を
1回の行列演算で結ぶだけ）のモデルだった。本ノートブックでは、同じデータに**隠れ層を1つ
追加**し、2層のニューラルネットワークに拡張する。5章と全く同じ15件の学習データ・6件の
評価データを使い、「層を増やすと何が変わるか／変わらないか」を実際の数値で確認する。

## データ（5章と同一）

学習15件・評価6件とも、5章のJSデモ（`05-1VectorMatrix.html`）と全く同じビット列である。

In [ ]:
import numpy as np

TRAINING_RAW = [
("0_01.txt", ["0000000000000000","0000000000000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000000000000","0000000000000000"]),
("0_02.txt", ["0000000000000000","0000000000000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000000000000","0000000000000000"]),
("0_03.txt", ["0000000000000000","0000000000000000","0000000000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000001000000000","0000000000000000","0000000000000000","0000000000000000"]),
("0_04.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000001000000","0000000000000000","0000000000000000","0000000000000000"]),
("0_05.txt", ["0000000000000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000100000000","0000000000000000","0000000000000000"]),
("1_01.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0011111111111100","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("1_02.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0011111111111100","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("1_03.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0001111111111000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("1_04.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0001111111111000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("1_05.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0111111111111110","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("2_01.txt", ["0000000000000000","0000000000000000","0000000000000100","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000100000000000","0001000000000000","0010000000000000","0000000000000000","0000000000000000"]),
("2_02.txt", ["0000000000000000","0000000000000000","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000100000000000","0001000000000000","0010000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("2_03.txt", ["0000000000000000","0000000000000010","0000000000000100","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000100000000000","0001000000000000","0010000000000000","0000000000000000","0000000000000000"]),
("2_04.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000100","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000100000000000","0001000000000000","0000000000000000","0000000000000000"]),
("2_05.txt", ["0000000000000000","0000000000000000","0000000000000010","0000000000000100","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000100000000000","0001000000000000","0000000000000000","0000000000000000"]),
]

EVAL_RAW = [
("0_near.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000010000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("0_far.txt",  ["0000000000000000","0000000000000000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000001000","0000000000000000","0000000000000000"]),
("1_near.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000111111110000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("1_far.txt",  ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0011111111111100","0000000000000000","0000000000000000"]),
("2_near.txt", ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000001000000000","0000010000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000"]),
("2_far.txt",  ["0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000000","0000000000000010","0000000000000100","0000000000001000","0000000000010000","0000000000100000","0000000001000000","0000000010000000","0000000100000000","0000000000000000","0000000000000000"]),
]

def to_bits(rows):
    return [1.0 if ch == "1" else 0.0 for row in rows for ch in row]

X = np.array([to_bits(rows) for _, rows in TRAINING_RAW])
y = np.array([int(name[0]) for name, _ in TRAINING_RAW])
targets = np.zeros((len(TRAINING_RAW), 3))
for i, c in enumerate(y):
    targets[i, c] = 1.0

Xe = np.array([to_bits(rows) for _, rows in EVAL_RAW])
ye = np.array([int(name[0]) for name, _ in EVAL_RAW])
names_e = [name.replace(".txt", "") for name, _ in EVAL_RAW]

print("X.shape =", X.shape, " targets.shape =", targets.shape, " Xe.shape =", Xe.shape)


## A) 比較用：5章と同じ1層のみのモデル

ゼロ初期化・2乗誤差（`Loss = Σ(prediction - target)²`）、5章と全く同じ式である。

In [ ]:
def train_1layer(epochs=600, rate=0.01):
    W = np.zeros((3, 256))
    b = np.zeros(3)
    for epoch in range(epochs):
        prediction = X @ W.T + b
        diff = prediction - targets
        loss = np.sum(diff ** 2)
        gradient = 2 * diff
        W -= rate * (gradient.T @ X)
        b -= rate * np.sum(gradient, axis=0)
    return W, b, loss

def predict_1layer(Xq, W, b):
    return np.argmax(Xq @ W.T + b, axis=1)

W, b, loss1 = train_1layer()
pred_train1, pred_eval1 = predict_1layer(X, W, b), predict_1layer(Xe, W, b)
print("[1層] 学習Loss=%.4f  学習データ正解=%d/15  評価データ正解=%d/6" %
      (loss1, (pred_train1 == y).sum(), (pred_eval1 == ye).sum()))
for name, p, t in zip(names_e, pred_eval1, ye):
    print(f"  {name:8s} 予測={p} 正解={t}  {'OK' if p==t else 'NG'}")


## B) 隠れ層を1つ持つ2層モデル

- 1層目：256入力 → H個の隠れユニット、ReLUで非線形変換する
- 2層目：H個の隠れユニット → 3クラス分のスコア

**なぜ間に非線形関数（ReLU）を挟む必要があるのか**が重要な点である。もしReLUを挟まず、
ただの行列演算だけを2回重ねると、

```
Z2 = (X @ W1.T + b1) @ W2.T + b2 = X @ (W1.T @ W2.T) + (b1 @ W2.T + b2)
```

のように、結局「Xに何か1つの行列を掛けて、何か1つのベクトルを足す」という**1層の式と
数学的に同じ形に潰れてしまう**。何層重ねても、間に非線形関数を挟まなければ1層分の
表現力しか持てない。ReLU（`max(0, x)`）のような非線形関数を挟むことで、初めて「層を
重ねる意味」が生まれる。

**逆伝播（連鎖律を2層ぶん繋げるだけ。1章・5章と考え方は同じ）**

```
dLoss/dZ2 = 2*(Z2-targets)                … 5章と同じ式
dLoss/dW2 = dLoss/dZ2.T @ A1
dLoss/dA1 = dLoss/dZ2 @ W2                … ここが新しい連鎖律の1段
dLoss/dZ1 = dLoss/dA1 * (Z1>0)            … ReLUの微分(0か1)を掛けるだけ
dLoss/dW1 = dLoss/dZ1.T @ X
```

In [ ]:
def relu(z):
    return np.maximum(z, 0)

def train_2layer(H=8, epochs=3000, rate=0.01, seed=0, zero_init=False):
    if zero_init:
        W1 = np.zeros((H, 256))
        W2 = np.zeros((3, H))
    else:
        rng = np.random.default_rng(seed)
        W1 = rng.normal(0, 0.1, size=(H, 256))
        W2 = rng.normal(0, 0.1, size=(3, H))
    b1 = np.zeros(H)
    b2 = np.zeros(3)

    for epoch in range(epochs):
        Z1 = X @ W1.T + b1
        A1 = relu(Z1)
        Z2 = A1 @ W2.T + b2

        diff = Z2 - targets
        loss = np.sum(diff ** 2)

        g2 = 2 * diff
        W2_grad = g2.T @ A1
        b2_grad = np.sum(g2, axis=0)

        dA1 = g2 @ W2
        dZ1 = dA1 * (Z1 > 0)
        W1_grad = dZ1.T @ X
        b1_grad = np.sum(dZ1, axis=0)

        W1 -= rate * W1_grad; b1 -= rate * b1_grad
        W2 -= rate * W2_grad; b2 -= rate * b2_grad

    return W1, b1, W2, b2, loss

def predict_2layer(Xq, W1, b1, W2, b2):
    return np.argmax(relu(Xq @ W1.T + b1) @ W2.T + b2, axis=1)

W1, b1, W2, b2, loss2 = train_2layer(H=8, zero_init=False)
pred_train2, pred_eval2 = predict_2layer(X, W1, b1, W2, b2), predict_2layer(Xe, W1, b1, W2, b2)
print("[2層,H=8] 学習Loss=%.4f  学習データ正解=%d/15  評価データ正解=%d/6" %
      (loss2, (pred_train2 == y).sum(), (pred_eval2 == ye).sum()))
for name, p, t in zip(names_e, pred_eval2, ye):
    print(f"  {name:8s} 予測={p} 正解={t}  {'OK' if p==t else 'NG'}")


## C) 初期値のワナ：隠れ層をゼロ初期化すると何が起きるか

2章で「初期値」が学習に影響する要素の1つとして触れたが、隠れ層ではこれが**致命的な
問題**になる。1層目の重みを全部ゼロにすると、どの隠れユニットも「同じ入力×同じ重み(0)」
という全く同じ計算をすることになり、勾配も全ユニットで完全に同じ値になる。結果、
**何回更新しても、H個の隠れユニットが最後まで全部同じ値のまま**になってしまう
（これを「対称性が壊れない」問題と呼ぶ）。実際に確認する。

In [ ]:
W1z, b1z, W2z, b2z, lossz = train_2layer(H=8, zero_init=True)
pred_trainz, pred_evalz = predict_2layer(X, W1z, b1z, W2z, b2z), predict_2layer(Xe, W1z, b1z, W2z, b2z)
print("[2層,ゼロ初期化] 学習Loss=%.4f  学習データ正解=%d/15  評価データ正解=%d/6" %
      (lossz, (pred_trainz == y).sum(), (pred_evalz == ye).sum()))
print("隠れユニットが全部同じ値になっているか:", np.allclose(W1z, W1z[0]))


## まとめ

| | 学習データ(15件) | 評価データ(6件) |
|---|---|---|
| A) 1層のみ(5章と同じ) | 15/15 | 3/6（near全部OK、far全部NG） |
| B) 2層、隠れ層H=8、乱数初期化 | 15/15 | 4/6（near全部OK、farのうち1件改善） |
| C) 2層、隠れ層H=8、ゼロ初期化 | 5/15 | 2/6（対称性が壊れず、学習データすら解けない） |

隠れ層を1つ足すだけで、評価データの正解数が3/6→4/6に改善した。ただし「far」（学習データ
から離れた位置の図形）のうち2件はまだ解けておらず、**層を増やすだけでは「位置がずれても
同じ形として認識する」という能力は十分には得られない**ことも分かる。これは全結合層
（Dense/Linear層）の構造上の限界であり、位置のずれに強い認識をするには、次に学ぶべき
CNN（畳み込みニューラルネットワーク）のような、別の仕組みが必要になる。

また(C)の結果は、2章で触れた「初期値」の重要性が、層を重ねるほど顕著な問題になることを
具体的に示している。